# FF-FP Gap Trajectories: Backbone Scaling Comparison

Comparing contrastive gap evolution across architectures:
- **12L H=1024** (154M params, 6.9 step/s) — 200k search + 2M full
- **16L H=1024** (204M params, 5.4 step/s) — 200k search only
- **20L H=1024** (255M params, 4.4 step/s) — 200k search + 2.3M full (lr=5.4e-5)

All use: GRU encoder, nhead=8, FFN 4x, GELU, depthwise conv k=3, bs=8.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import re

def parse_gap_log(path, sample_every=1):
    """Extract (step, gap) pairs from a training log."""
    steps, gaps = [], []
    with open(path) as f:
        for line in f:
            m = re.search(r'Step (\d+)\].*gap=([-\d.]+)', line)
            if m:
                steps.append(int(m.group(1)))
                gaps.append(float(m.group(2)))
    steps, gaps = np.array(steps), np.array(gaps)
    if sample_every > 1:
        idx = np.arange(0, len(steps), sample_every)
        steps, gaps = steps[idx], gaps[idx]
    return steps, gaps

# Base directory — adjust if running elsewhere
BASE = os.path.expanduser("~/workspaces/contrastive-forecasting")
SCALE = os.path.join(BASE, "scaling_search_logs")

In [ ]:
# Load all trajectories
data = {}

# 200k search runs
data["12L (200k)"] = parse_gap_log(os.path.join(SCALE, "scaling_12L_H1024_baseline.log"))
data["16L (200k)"] = parse_gap_log(os.path.join(SCALE, "scaling_16L_H1024.log"))
data["20L (200k)"] = parse_gap_log(os.path.join(SCALE, "scaling_20L_H1024.log"))

# Long runs — 12L 2M (phase4 + resumed)
s4, g4 = parse_gap_log(os.path.join(BASE, "arch_search_phase4.log"), sample_every=5)
s2m, g2m = parse_gap_log(os.path.join(BASE, "v2_2M_training_resumed.log"), sample_every=5)
# Phase4 steps are absolute, resumed run steps restart from 1 — offset by 500k
# Actually the resumed run's step counter reset, so these are independent
data["12L (500k phase4)"] = (s4, g4)
data["12L (2M resumed)"] = (s2m, g2m)

# 20L long run
s20, g20 = parse_gap_log(os.path.join(SCALE, "scaling_20L_2M_lr54.log"), sample_every=5)
# This run starts at step 201k (resumed from 200k checkpoint)
data["20L (2.3M, lr=5.4e-5)"] = (s20, g20)

for name, (s, g) in data.items():
    print(f"{name}: {len(s)} points, steps {s[0]}-{s[-1]}, peak gap {g.max():.4f}")

## Plot 1: 200k Comparison (12L vs 16L vs 20L)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

colors = {"12L": "#1f77b4", "16L": "#ff7f0e", "20L": "#2ca02c"}

for name in ["12L (200k)", "16L (200k)", "20L (200k)"]:
    s, g = data[name]
    label_short = name.split(" ")[0]
    # Smooth with rolling mean for clarity
    window = 5
    g_smooth = np.convolve(g, np.ones(window)/window, mode='valid')
    s_smooth = s[window-1:]
    ax.plot(s_smooth / 1000, g_smooth, label=name, color=colors[label_short], alpha=0.8)
    ax.plot(s / 1000, g, color=colors[label_short], alpha=0.15)

ax.set_xlabel("Steps (k)", fontsize=12)
ax.set_ylabel("FF-FP Gap", fontsize=12)
ax.set_title("Scaling Comparison at 200k Steps (H=1024, bs=8, lr=7e-5)", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 200)
ax.set_ylim(-0.02, 0.18)
plt.tight_layout()
plt.savefig("../report/images/gap_200k_comparison.png", dpi=150)
plt.show()

## Plot 2: Full Training (12L vs 20L, up to 2M+ steps)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# 12L: combine phase4 (0-500k) and resumed (step counter reset, ~0-2M)
# The resumed run's step counter reset to 1, these are independent training segments
s_12_p4, g_12_p4 = data["12L (500k phase4)"]
s_12_2m, g_12_2m = data["12L (2M resumed)"]
# Plot phase4 first, then resumed (step counter independent)
ax.plot(s_12_p4 / 1e6, g_12_p4, color="#1f77b4", alpha=0.15)
ax.plot(s_12_2m / 1e6, g_12_2m, color="#1f77b4", alpha=0.15)
# Smoothed
w = 10
g_p4_s = np.convolve(g_12_p4, np.ones(w)/w, mode='valid')
g_2m_s = np.convolve(g_12_2m, np.ones(w)/w, mode='valid')
ax.plot(s_12_p4[w-1:] / 1e6, g_p4_s, color="#1f77b4", alpha=0.8, label="12L (phase4, 500k)")
ax.plot(s_12_2m[w-1:] / 1e6, g_2m_s, color="#1f77b4", alpha=0.8, linestyle="--", label="12L (2M resumed, lr=7e-5)")

# 20L: 200k search + 2.3M continuation
s_20_200k, g_20_200k = data["20L (200k)"]
s_20_long, g_20_long = data["20L (2.3M, lr=5.4e-5)"]
ax.plot(s_20_200k / 1e6, g_20_200k, color="#2ca02c", alpha=0.15)
ax.plot(s_20_long / 1e6, g_20_long, color="#2ca02c", alpha=0.15)
w2 = 5
g_20_200k_s = np.convolve(g_20_200k, np.ones(w2)/w2, mode='valid')
g_20_long_s = np.convolve(g_20_long, np.ones(w)/w, mode='valid')
ax.plot(s_20_200k[w2-1:] / 1e6, g_20_200k_s, color="#2ca02c", alpha=0.8, label="20L (200k search, lr=7e-5)")
ax.plot(s_20_long[w-1:] / 1e6, g_20_long_s, color="#2ca02c", alpha=0.8, linestyle="--", label="20L (2.3M, lr=5.4e-5)")

ax.axhline(y=0.203, color="gray", linestyle=":", alpha=0.5, label="12L 2M peak (0.203)")
ax.set_xlabel("Steps (M)", fontsize=12)
ax.set_ylabel("FF-FP Gap", fontsize=12)
ax.set_title("Gap Trajectory: 12L vs 20L (H=1024)", fontsize=13)
ax.legend(fontsize=10, loc="lower right")
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.02, 0.22)
plt.tight_layout()
plt.savefig("../report/images/gap_full_training.png", dpi=150)
plt.show()

## Plot 3: Gap at Matched Steps (table view)

In [ ]:
# Gap at matched step milestones
milestones = [20, 50, 100, 150, 200]  # in thousands

def gap_at_step(steps, gaps, target_k):
    """Find gap closest to target step (in thousands)."""
    target = target_k * 1000
    idx = np.argmin(np.abs(steps - target))
    if abs(steps[idx] - target) < 5000:
        return f"{gaps[idx]:.4f}"
    return "—"

print(f"{'Step':>8s}  {'12L':>8s}  {'16L':>8s}  {'20L':>8s}")
print("-" * 38)
for m in milestones:
    s12, g12 = data["12L (200k)"]
    s16, g16 = data["16L (200k)"]
    s20, g20 = data["20L (200k)"]
    print(f"{m:>6d}k  {gap_at_step(s12, g12, m):>8s}  {gap_at_step(s16, g16, m):>8s}  {gap_at_step(s20, g20, m):>8s}")

# Extended milestones for long runs
print()
print("Extended training:")
print(f"{'Step':>8s}  {'12L (2M)':>10s}  {'20L (lr54)':>12s}")
print("-" * 36)
s12_l, g12_l = data["12L (2M resumed)"]
s20_l, g20_l = data["20L (2.3M, lr=5.4e-5)"]
for m in [500, 1000, 1500, 2000]:
    print(f"{m:>6d}k  {gap_at_step(s12_l, g12_l, m):>10s}  {gap_at_step(s20_l, g20_l, m):>12s}")